# Welchen Kalender kennt das Stromnetz?

Aus dem kantonalen Stromverbrauch von Basel-Stadt (Viertelstundenwerte,
IWB über Open Data Basel-Stadt, Datensatz 100233) werden je Kalenderjahr
die zehn verbrauchsärmsten Werktage (Montag bis Freitag) ermittelt. Das
Ergebnis landet in `data/stillste-tage.json` und speist `index.html`.

Warum Werktage: Über alle Tage gerechnet sind die zehn stillsten Tage
jedes Jahres ausnahmslos Sonntage (Abschnitt 6). Der Wochenrhythmus
überdeckt den Kalender. Erst ohne Wochenenden zeigt sich, welche Tage das
Netz sonst noch kennt.

Nur Standardbibliothek: `csv`, `datetime`, `statistics`, `json`, `collections`.
Die Analyse liest ausschliesslich die lokale Datei `data/verbrauch.csv`.

Regeln, die hier eingehalten werden:

* Datei aufsteigend sortieren (sie ist absteigend geliefert).
* `Wochentag` ist nullbasiert ab Montag; er wird aus dem Zeitstempel
  nachgerechnet, bei Abweichung bricht das Notebook ab.
* Tage werden nie über die Tagessumme verglichen, sondern über
  Mittelwert der Intervalle × 96 (Zeitumstellung: 92 oder 100 Intervalle).
* Tage mit anderer Intervallzahl als 92, 96, 100 sowie erster und letzter
  Tag der Datei werden verworfen.
* Rangfolge nur innerhalb eines Jahres, zusätzlich Wert in Prozent des
  Medians der Werktage des Jahres.
* Die Teilspalten `Grundversorgte Kunden` / `Freie Kunden` gelten erst ab
  1. September 2020 als vorhanden; früher werden sie als fehlend behandelt.
* Drei Zeitstempel sind laut Herausgeber interpoliert und werden markiert.

In [1]:
import csv
import datetime
import json
import statistics
from collections import defaultdict, Counter

# Der Rohstand heisst data/verbrauch.csv. Liegt er noch unter dem Portalnamen
# data/100233.csv (siehe README, Offene Punkte), wird dieser gelesen.
try:
    open("data/verbrauch.csv", encoding="utf-8-sig").close()
    CSV_PFAD = "data/verbrauch.csv"
except FileNotFoundError:
    CSV_PFAD = "data/100233.csv"
JSON_PFAD = "data/stillste-tage.json"
ANZAHL_JE_JAHR = 10
WERKTAGE = range(0, 5)   # Montag = 0 bis Freitag = 4
ALLE_TAGE = range(0, 7)
GUELTIGE_INTERVALLZAHLEN = {92, 96, 100}
TEILSPALTEN_AB = datetime.date(2020, 9, 1)
INTERPOLIERT = {
    "2014-01-01T00:00:00+01:00",
    "2015-11-01T00:00:00+01:00",
    "2019-10-01T00:00:00+02:00",
}
ERWARTETE_KOPFZEILE = [
    "Start der Messung", "Start der Messung (Text)", "Stromverbrauch",
    "Grundversorgte Kunden", "Freie Kunden", "Jahr", "Monat", "Tag",
    "Wochentag", "Tag des Jahres", "Quartal", "Woche des Jahres",
]

## 1. Einlesen und aufsteigend sortieren

Die Datei beginnt mit einer BOM (`utf-8-sig` entfernt sie). Der Zeitstempel
trägt den Offset (`+01:00` / `+02:00`); `datetime.fromisoformat` versteht ihn.
Der Kalendertag eines Intervalls ist der lokale Tag im Zeitstempel.

In [2]:
zeilen = []
with open(CSV_PFAD, encoding="utf-8-sig", newline="") as f:
    leser = csv.DictReader(f, delimiter=";")
    assert leser.fieldnames == ERWARTETE_KOPFZEILE, leser.fieldnames
    for zeile in leser:
        zeilen.append(zeile)

for z in zeilen:
    z["ts"] = datetime.datetime.fromisoformat(z["Start der Messung"])
    z["wert"] = float(z["Stromverbrauch"])

zeilen.sort(key=lambda z: z["ts"])

print("Datei:", CSV_PFAD)
print("Zeilen:", len(zeilen))
print("Erste Messung:", zeilen[0]["Start der Messung"])
print("Letzte Messung:", zeilen[-1]["Start der Messung"])

Datei: data/verbrauch.csv
Zeilen: 515267
Erste Messung: 2012-01-01T00:15:00+01:00
Letzte Messung: 2026-09-11T23:45:00+02:00


## 2. Wochentag nachrechnen

Kontrolle der Konvention Montag = 0 gegen den Zeitstempel. Eine einzige
Abweichung bricht ab.

In [3]:
abweichungen = [
    z["Start der Messung"] for z in zeilen
    if z["ts"].weekday() != int(z["Wochentag"])
]
if abweichungen:
    raise SystemExit(
        f"Wochentag-Konvention verletzt bei {len(abweichungen)} Zeilen, "
        f"z. B. {abweichungen[:3]}"
    )
stichprobe = zeilen[-1]
print("Abweichungen:", len(abweichungen))
print("Stichprobe:", stichprobe["Start der Messung"],
      "weekday() =", stichprobe["ts"].weekday(),
      "Spalte =", stichprobe["Wochentag"])

Abweichungen: 0
Stichprobe: 2026-09-11T23:45:00+02:00 weekday() = 4 Spalte = 4


## 3. Intervalle je Kalendertag bündeln

Pro Tag: Anzahl Intervalle, Mittelwert × 96 als vergleichbarer Tageswert,
Markierung, falls ein interpolierter Zeitstempel enthalten ist. Für die
Teilspalten dasselbe, aber nur ab 1. September 2020 und nur, wenn beide
Felder in jedem Intervall des Tages gefüllt sind.

In [4]:
je_tag = defaultdict(list)
for z in zeilen:
    je_tag[z["ts"].date()].append(z)

tage = []
for datum in sorted(je_tag):
    intervalle = je_tag[datum]
    n = len(intervalle)
    gesamt = [z["wert"] for z in intervalle]
    eintrag = {
        "datum": datum,
        "intervalle": n,
        "wert": statistics.fmean(gesamt) * 96,
        "interpoliert": any(z["Start der Messung"] in INTERPOLIERT for z in intervalle),
        "grund": None,
        "frei": None,
        # Intervallwerte für das Tagesprofil auf der Seite
        "profil_wert": gesamt,
        "profil_grund": None,
        "profil_frei": None,
    }
    if datum >= TEILSPALTEN_AB and all(
        z["Grundversorgte Kunden"] != "" and z["Freie Kunden"] != "" for z in intervalle
    ):
        eintrag["profil_grund"] = [float(z["Grundversorgte Kunden"]) for z in intervalle]
        eintrag["profil_frei"] = [float(z["Freie Kunden"]) for z in intervalle]
        eintrag["grund"] = statistics.fmean(eintrag["profil_grund"]) * 96
        eintrag["frei"] = statistics.fmean(eintrag["profil_frei"]) * 96
    tage.append(eintrag)

print("Kalendertage in der Datei:", len(tage))
print("Verteilung der Intervallzahlen:", dict(Counter(t["intervalle"] for t in tage)))

Kalendertage in der Datei: 5368
Verteilung der Intervallzahlen: {95: 1, 96: 5352, 92: 15}


## 4. Unvollständige Tage verwerfen

Erster und letzter Tag der Datei sind angeschnitten. Alle übrigen Tage mit
einer Intervallzahl ausserhalb von 92/96/100 fliegen ebenfalls raus.

Beobachtung: In diesem Stand hat kein Tag 100 Intervalle. Am Tag der
Umstellung auf Winterzeit fehlt die doppelte Stunde, der Tag hat 96
Intervalle. Die März-Tage haben wie erwartet 92.

In [5]:
erster, letzter = tage[0], tage[-1]
verworfen = [erster, letzter] + [
    t for t in tage[1:-1] if t["intervalle"] not in GUELTIGE_INTERVALLZAHLEN
]
gueltig = [t for t in tage[1:-1] if t["intervalle"] in GUELTIGE_INTERVALLZAHLEN]

for t in verworfen:
    print("verworfen:", t["datum"], "Intervalle:", t["intervalle"])
print("Gültige Tage:", len(gueltig))
assert all(t["intervalle"] in GUELTIGE_INTERVALLZAHLEN for t in gueltig)

verworfen: 2012-01-01 Intervalle: 95
verworfen: 2026-09-11 Intervalle: 96
Gültige Tage: 5366


## 5. Zehn stillste Werktage je Jahr, relativ zum Werktagsmedian

Nur Montag bis Freitag. Der Median wird über alle gültigen Werktage des
Jahres gebildet; ein normaler Werktag liegt damit nahe 100 %. Für die
Seite bekommt jeder gelistete Tag sein Viertelstundenprofil und die Werte
seiner Woche (Montag bis Sonntag, in Prozent desselben Medians), jedes
Jahr das Median-Profil eines Werktags als Vergleich. Das laufende
Jahr 2026 ist unvollständig (bis 10. September); sein Median und seine
Rangliste sind deshalb nur bedingt mit ganzen Jahren vergleichbar.

In [6]:
tag_je_datum = {t["datum"]: t for t in gueltig}

def wochenkontext(t, schluessel, median):
    """Montag bis Sonntag der Woche des Tages, je in Prozent des Medians."""
    montag = t["datum"] - datetime.timedelta(days=t["datum"].weekday())
    woche = []
    for i in range(7):
        d = montag + datetime.timedelta(days=i)
        n = tag_je_datum.get(d)
        woche.append({
            "datum": d.isoformat(),
            "prozent": round(100 * n[schluessel] / median, 1) if n and n[schluessel] is not None else None,
        })
    return woche

def rangliste(tage_liste, schluessel, wochentage=WERKTAGE, mit_profil=True):
    """Zehn niedrigste Tage je Jahr nach `schluessel`, beschränkt auf
    `wochentage`, mit Prozent des Medians dieser Tage im Jahr. Dazu je Tag
    das Viertelstundenprofil und die Woche, je Jahr das Median-Profil eines
    Tages dieser Wochentage (nur Tage mit 96 Intervallen)."""
    je_jahr = defaultdict(list)
    for t in tage_liste:
        if t[schluessel] is not None and t["datum"].weekday() in wochentage:
            je_jahr[t["datum"].year].append(t)
    ergebnis = []
    for jahr in sorted(je_jahr):
        liste = je_jahr[jahr]
        median = statistics.median(t[schluessel] for t in liste)
        stillste = sorted(liste, key=lambda t: t[schluessel])[:ANZAHL_JE_JAHR]
        eintrag = {
            "jahr": jahr,
            "tage": len(liste),
            "vollstaendig": len(liste) >= 360 * len(wochentage) // 7,
            "erster_tag": liste[0]["datum"].isoformat(),
            "letzter_tag": liste[-1]["datum"].isoformat(),
            "median": round(median, 1),
            "stillste": [
                {
                    "rang": i + 1,
                    "datum": t["datum"].isoformat(),
                    "wochentag": t["datum"].weekday(),
                    "wert": round(t[schluessel], 1),
                    "prozent": round(100 * t[schluessel] / median, 1),
                    "intervalle": t["intervalle"],
                    "interpoliert": t["interpoliert"],
                }
                for i, t in enumerate(stillste)
            ],
        }
        if mit_profil:
            volle = [t["profil_" + schluessel] for t in liste if t["intervalle"] == 96]
            eintrag["profil_typisch"] = [
                round(statistics.median(p[i] for p in volle)) for i in range(96)
            ]
            for t, e in zip(stillste, eintrag["stillste"]):
                e["profil"] = [round(v) for v in t["profil_" + schluessel]]
                e["woche"] = wochenkontext(t, schluessel, median)
        ergebnis.append(eintrag)
    return ergebnis

werktage_je_jahr = rangliste(gueltig, "wert")

WOCHENTAGE = ["Mo", "Di", "Mi", "Do", "Fr", "Sa", "So"]
for jahr in werktage_je_jahr:
    print(f"\n{jahr['jahr']}  ({jahr['tage']} Werktage, Werktagsmedian {jahr['median']:.0f})")
    for t in jahr["stillste"]:
        markierung = "  [interpolierter Zeitstempel]" if t["interpoliert"] else ""
        print(f"  {t['rang']:2d}. {t['datum']} {WOCHENTAGE[t['wochentag']]}"
              f"  {t['wert']:9.0f}  {t['prozent']:5.1f} %{markierung}")


2012  (261 Werktage, Werktagsmedian 4597281)
   1. 2012-12-25 Di    3348907   72.8 %
   2. 2012-12-26 Mi    3395774   73.9 %
   3. 2012-04-09 Mo    3418542   74.4 %
   4. 2012-05-17 Do    3447051   75.0 %
   5. 2012-05-28 Mo    3474291   75.6 %
   6. 2012-04-06 Fr    3484896   75.8 %
   7. 2012-05-01 Di    3576118   77.8 %
   8. 2012-08-01 Mi    3665792   79.7 %
   9. 2012-12-31 Mo    3762022   81.8 %
  10. 2012-12-24 Mo    3791854   82.5 %

2013  (261 Werktage, Werktagsmedian 4352400)
   1. 2013-04-01 Mo    3228723   74.2 %
   2. 2013-12-25 Mi    3286697   75.5 %
   3. 2013-05-20 Mo    3320416   76.3 %
   4. 2013-12-26 Do    3329507   76.5 %
   5. 2013-03-29 Fr    3329630   76.5 %
   6. 2013-01-01 Di    3345526   76.9 %
   7. 2013-05-09 Do    3424944   78.7 %
   8. 2013-12-31 Di    3625476   83.3 %
   9. 2013-05-01 Mi    3645875   83.8 %
  10. 2013-08-01 Do    3651798   83.9 %

2014  (261 Werktage, Werktagsmedian 4198989)
   1. 2014-04-21 Mo    3142984   74.9 %
   2. 2014-05-29 Do   

## 6. Warum nicht einfach alle Tage? Der Sonntagsbefund

Dieselbe Rangliste über alle Wochentage. Sie zeigt, warum die Seite
Werktage braucht: Es kommen fast nur Sonntage heraus.

In [7]:
alle_je_jahr = rangliste(gueltig, "wert", ALLE_TAGE, mit_profil=False)
alle_plaetze = sum(len(j["stillste"]) for j in alle_je_jahr)
alle_sonntage = sum(1 for j in alle_je_jahr for t in j["stillste"] if t["wochentag"] == 6)
alle_werktage = [(t["datum"], WOCHENTAGE[t["wochentag"]]) for j in alle_je_jahr
                 for t in j["stillste"] if t["wochentag"] < 5]
print(f"Alle Tage: {alle_plaetze} Ranglistenplätze, davon Sonntage: {alle_sonntage}")
print("Rang 1 je Jahr:")
for j in alle_je_jahr:
    t = j["stillste"][0]
    print(f"  {j['jahr']}: {t['datum']} {WOCHENTAGE[t['wochentag']]}  {t['prozent']:.1f} %")
print("Werktage in dieser Liste:", alle_werktage)

Alle Tage: 150 Ranglistenplätze, davon Sonntage: 133
Rang 1 je Jahr:
  2012: 2012-07-22 So  71.9 %
  2013: 2013-05-19 So  73.7 %
  2014: 2014-04-20 So  70.2 %
  2015: 2015-12-27 So  72.3 %
  2016: 2016-03-27 So  72.5 %
  2017: 2017-04-16 So  74.0 %
  2018: 2018-04-01 So  74.6 %
  2019: 2019-04-21 So  75.6 %
  2020: 2020-04-12 So  77.0 %
  2021: 2021-05-23 So  74.7 %
  2022: 2022-04-17 So  74.1 %
  2023: 2023-04-09 So  75.9 %
  2024: 2024-03-31 So  75.3 %
  2025: 2025-08-03 So  78.2 %
  2026: 2026-04-05 So  77.3 %
Werktage in dieser Liste: [('2012-12-25', 'Di'), ('2013-04-01', 'Mo'), ('2015-12-25', 'Fr'), ('2016-01-01', 'Fr'), ('2017-12-25', 'Mo'), ('2019-12-25', 'Mi'), ('2019-12-26', 'Do'), ('2020-04-13', 'Mo'), ('2021-05-24', 'Mo'), ('2022-04-18', 'Mo'), ('2023-12-25', 'Mo'), ('2024-04-01', 'Mo'), ('2024-03-29', 'Fr'), ('2025-06-09', 'Mo'), ('2026-04-06', 'Mo')]


## 7. Sind die stillsten Werktage die Feiertage?

Die Feiertage stehen nicht in den Daten. Hier werden von Hand die
gesetzlichen Feiertage von Basel-Stadt gebildet (Neujahr, Karfreitag,
Ostermontag, 1. Mai, Auffahrt, Pfingstmontag, 1. August, Weihnachten,
Stephanstag; Ostern nach der Gauss-Formel), dazu Heiligabend und Silvester,
die sicher am Datum hängen. Die Liste greift nicht in die Auswahl der Tage
ein. Sie wird als eigene Datei `data/feiertage.json` abgelegt und liefert
die Namen, die die Seite beim Aufdecken zeigt. Tage ohne Eintrag zeigen
nur Datum und Wochentag; es werden keine Namen geraten.

In [8]:
def ostersonntag(jahr):
    """Gauss/Anonymous Gregorian algorithm."""
    a = jahr % 19
    b, c = divmod(jahr, 100)
    d, e = divmod(b, 4)
    f = (b + 8) // 25
    g = (b - f + 1) // 3
    h = (19 * a + b - d - g + 15) % 30
    i, k = divmod(c, 4)
    l = (32 + 2 * e + 2 * i - h - k) % 7
    m = (a + 11 * h + 22 * l) // 451
    monat, tag = divmod(h + l - 7 * m + 114, 31)
    return datetime.date(jahr, monat, tag + 1)

def feiertage_basel(jahr):
    """Gesetzliche Feiertage Basel-Stadt."""
    ostern = ostersonntag(jahr)
    return {
        datetime.date(jahr, 1, 1): "Neujahr",
        ostern - datetime.timedelta(days=2): "Karfreitag",
        ostern + datetime.timedelta(days=1): "Ostermontag",
        datetime.date(jahr, 5, 1): "Tag der Arbeit",
        ostern + datetime.timedelta(days=39): "Auffahrt",
        ostern + datetime.timedelta(days=50): "Pfingstmontag",
        datetime.date(jahr, 8, 1): "Bundesfeiertag",
        datetime.date(jahr, 12, 25): "Weihnachten",
        datetime.date(jahr, 12, 26): "Stephanstag",
    }

def feiertagsnamen(jahr):
    """Gesetzliche Feiertage plus Heiligabend und Silvester, für die Anzeige."""
    namen = dict(feiertage_basel(jahr))
    namen[datetime.date(jahr, 12, 24)] = "Heiligabend"
    namen[datetime.date(jahr, 12, 31)] = "Silvester"
    return namen

FEIERTAGE_PFAD = "data/feiertage.json"
feiertage_liste = [
    {"datum": d.isoformat(), "name": n, "gesetzlich": d in feiertage_basel(d.year)}
    for jahr in range(tage[0]["datum"].year, tage[-1]["datum"].year + 1)
    for d, n in sorted(feiertagsnamen(jahr).items())
]
with open(FEIERTAGE_PFAD, "w", encoding="utf-8") as f:
    json.dump({
        "hinweis": "Von Hand ergänzt, nicht aus den Daten. Gesetzliche Feiertage Basel-Stadt, "
                   "Ostern nach der Gauss-Formel, dazu Heiligabend und Silvester.",
        "feiertage": feiertage_liste,
    }, f, ensure_ascii=False, indent=1)
print("geschrieben:", FEIERTAGE_PFAD, len(feiertage_liste), "Einträge")

def kategorie(datum):
    if datum in feiertage_basel(datum.year):
        return "gesetzlicher Feiertag"
    if (datum.month == 12 and datum.day >= 24) or (datum.month == 1 and datum.day <= 2):
        return "Werktag zwischen Weihnachten und Neujahr"
    ostern = ostersonntag(datum.year)
    if datum in (ostern + datetime.timedelta(days=40), ostern + datetime.timedelta(days=-3)):
        return "Brückentag (Freitag nach Auffahrt, Gründonnerstag)"
    return "anderer Werktag"

zaehler = Counter()
andere = []
for jahr in werktage_je_jahr:
    for t in jahr["stillste"]:
        d = datetime.date.fromisoformat(t["datum"])
        k = kategorie(d)
        zaehler[k] += 1
        if k == "anderer Werktag":
            andere.append(f"{t['datum']} {WOCHENTAGE[t['wochentag']]} {t['prozent']:.0f} %")

plaetze = sum(zaehler.values())
feiertage_in_liste = zaehler["gesetzlicher Feiertag"]
print(f"{plaetze} Ranglistenplätze {werktage_je_jahr[0]['jahr']}–{werktage_je_jahr[-1]['jahr']}")
for k, n in zaehler.most_common():
    print(f"  {n:3d}  {k}")
print("\nAndere Werktage:")
for a_ in andere:
    print("  ", a_)

print("\nFeiertage je Jahr in der Liste (von 9 gesetzlichen, soweit im Datenstand):")
feiertage_je_jahr = {}
for jahr in werktage_je_jahr:
    letzter = datetime.date.fromisoformat(jahr["letzter_tag"])
    ft = {d: n for d, n in feiertage_basel(jahr["jahr"]).items() if d <= letzter}
    an_werktagen = sum(1 for d in ft if d.weekday() < 5)
    treffer = [ft[datetime.date.fromisoformat(t["datum"])] for t in jahr["stillste"]
               if datetime.date.fromisoformat(t["datum"]) in ft]
    fehlend = [name for d, name in sorted(ft.items()) if d.weekday() < 5 and name not in treffer]
    feiertage_je_jahr[jahr["jahr"]] = (len(treffer), an_werktagen)
    print(f"  {jahr['jahr']}: {len(treffer)} von {an_werktagen} Feiertagen an Werktagen"
          + (f", nicht in der Liste: {', '.join(fehlend)}" if fehlend else ""))

geschrieben: data/feiertage.json 165 Einträge
150 Ranglistenplätze 2012–2026
  115  gesetzlicher Feiertag
   23  Werktag zwischen Weihnachten und Neujahr
    8  Brückentag (Freitag nach Auffahrt, Gründonnerstag)
    4  anderer Werktag

Andere Werktage:
   2020-04-03 Fr 87 %
   2023-07-31 Mo 90 %
   2026-06-05 Fr 92 %
   2026-07-24 Fr 92 %

Feiertage je Jahr in der Liste (von 9 gesetzlichen, soweit im Datenstand):
  2012: 8 von 8 Feiertagen an Werktagen
  2013: 9 von 9 Feiertagen an Werktagen
  2014: 9 von 9 Feiertagen an Werktagen
  2015: 7 von 7 Feiertagen an Werktagen
  2016: 7 von 7 Feiertagen an Werktagen
  2017: 8 von 8 Feiertagen an Werktagen
  2018: 8 von 9 Feiertagen an Werktagen, nicht in der Liste: Bundesfeiertag
  2019: 9 von 9 Feiertagen an Werktagen
  2020: 7 von 7 Feiertagen an Werktagen
  2021: 5 von 5 Feiertagen an Werktagen
  2022: 6 von 6 Feiertagen an Werktagen
  2023: 8 von 8 Feiertagen an Werktagen
  2024: 9 von 9 Feiertagen an Werktagen
  2025: 9 von 9 Feiertagen 

## 8. Wie still sind Feiertage und Betriebsferien im Vergleich?

Median des relativen Tageswerts je Kategorie über alle vollständigen
Jahre, jeweils in Prozent des Werktagsmedians des Jahres. Sonntage und
Samstage zum Vergleich.

In [9]:
je_kategorie = defaultdict(list)
werktagsmedian = {j["jahr"]: j["median"] for j in werktage_je_jahr}
vollstaendige_jahre = {j["jahr"] for j in werktage_je_jahr if j["vollstaendig"]}
for t in gueltig:
    d = t["datum"]
    if d.year not in vollstaendige_jahre:
        continue
    if d.weekday() == 6:
        k = "Ostersonntag" if d == ostersonntag(d.year) else "Sonntag"
    elif d.weekday() == 5:
        k = "Samstag"
    else:
        k = kategorie(d)
    je_kategorie[k].append(100 * t["wert"] / werktagsmedian[d.year])

print("Median des Tageswerts in Prozent des Werktagsmedians, vollständige Jahre:")
for k, werte in sorted(je_kategorie.items(), key=lambda kv: statistics.median(kv[1])):
    print(f"  {statistics.median(werte):5.1f} %  {k} (n = {len(werte)})")

Median des Tageswerts in Prozent des Werktagsmedians, vollständige Jahre:
   73.2 %  Ostersonntag (n = 14)
   78.8 %  gesetzlicher Feiertag (n = 110)
   79.4 %  Sonntag (n = 716)
   86.4 %  Samstag (n = 730)
   90.4 %  Werktag zwischen Weihnachten und Neujahr (n = 69)
   90.6 %  Brückentag (Freitag nach Auffahrt, Gründonnerstag) (n = 28)
  100.3 %  anderer Werktag (n = 3446)


## 9. Ausbaustufe: Grundversorgung und freier Markt getrennt (ab September 2020)

Dieselbe Werktagsregel je Teilspalte.

In [10]:
grund_je_jahr = rangliste(gueltig, "grund")
frei_je_jahr = rangliste(gueltig, "frei")

for name, liste in (("Grundversorgte Kunden", grund_je_jahr), ("Freie Kunden", frei_je_jahr)):
    print(f"\n=== {name} ===")
    for jahr in liste:
        print(f"{jahr['jahr']} ({jahr['tage']} Werktage, Median {jahr['median']:.0f}):",
              ", ".join(f"{t['datum']} {WOCHENTAGE[t['wochentag']]} {t['prozent']:.0f}%"
                        for t in jahr["stillste"][:5]), "…")


=== Grundversorgte Kunden ===
2020 (88 Werktage, Median 1717698): 2020-12-25 Fr 88%, 2020-09-07 Mo 88%, 2020-09-03 Do 89%, 2020-09-08 Di 89%, 2020-09-09 Mi 89% …
2021 (261 Werktage, Median 1538619): 2021-04-02 Fr 84%, 2021-05-24 Mo 86%, 2021-08-02 Mo 86%, 2021-07-21 Mi 87%, 2021-07-19 Mo 87% …
2022 (260 Werktage, Median 1485480): 2022-04-18 Mo 80%, 2022-05-26 Do 81%, 2022-08-01 Mo 81%, 2022-04-15 Fr 82%, 2022-06-06 Mo 82% …
2023 (260 Werktage, Median 1509397): 2023-05-29 Mo 80%, 2023-08-01 Di 80%, 2023-04-10 Mo 81%, 2023-07-31 Mo 82%, 2023-05-18 Do 84% …
2024 (262 Werktage, Median 1563649): 2024-05-20 Mo 83%, 2024-08-01 Do 85%, 2024-07-17 Mi 86%, 2024-04-01 Mo 86%, 2024-05-01 Mi 86% …
2025 (261 Werktage, Median 1555984): 2025-08-01 Fr 77%, 2025-05-29 Do 79%, 2025-06-09 Mo 80%, 2025-05-01 Do 81%, 2025-05-30 Fr 82% …
2026 (181 Werktage, Median 1550532): 2026-04-06 Mo 80%, 2026-05-25 Mo 81%, 2026-05-01 Fr 81%, 2026-07-21 Di 84%, 2026-07-24 Fr 84% …

=== Freie Kunden ===
2020 (88 Werktage

## 10. JSON schreiben

In [11]:
name_je_datum = {e["datum"]: e["name"] for e in feiertage_liste}
for liste in (werktage_je_jahr, grund_je_jahr, frei_je_jahr):
    for jahr in liste:
        for t in jahr["stillste"]:
            t["feiertag"] = name_je_datum.get(t["datum"])

ausgabe = {
    "quelle": "Kantonaler Stromverbrauch, IWB über Open Data Basel-Stadt, Datensatz 100233, CC BY 4.0",
    "stand": zeilen[-1]["ts"].date().isoformat(),
    "einheit": "vermutlich kWh je 15-Minuten-Intervall, laut Portal prüfen",
    "methode": "Nur Montag bis Freitag. Tageswert = Mittelwert der 15-Minuten-Intervalle × 96; Prozent = Tageswert / Median aller gültigen Werktage des Jahres",
    "anzahl_je_jahr": ANZAHL_JE_JAHR,
    "verworfene_tage": [
        {"datum": t["datum"].isoformat(), "intervalle": t["intervalle"]} for t in verworfen
    ],
    "interpolierte_zeitstempel": sorted(INTERPOLIERT),
    "feiertagsnamen": "von Hand ergänzt, siehe data/feiertage.json",
    "befund": {
        "werktage": {
            "ranglistenplaetze": plaetze,
            "jahre": len(werktage_je_jahr),
            "kategorien": dict(zaehler),
            "feiertage_je_jahr": {str(j): {"in_liste": n, "an_werktagen": m}
                                  for j, (n, m) in feiertage_je_jahr.items()},
        },
        "alle_tage": {
            "ranglistenplaetze": alle_plaetze,
            "davon_sonntage": alle_sonntage,
            "rang_1_je_jahr": {str(j["jahr"]): j["stillste"][0]["datum"] for j in alle_je_jahr},
        },
        "median_prozent_je_kategorie_vollstaendige_jahre": {
            k: round(statistics.median(w), 1) for k, w in je_kategorie.items()
        },
    },
    "gesamt": werktage_je_jahr,
    "segmente": {
        "grund": {"name": "Grundversorgte Kunden", "ab": TEILSPALTEN_AB.isoformat(), "jahre": grund_je_jahr},
        "frei": {"name": "Freie Kunden", "ab": TEILSPALTEN_AB.isoformat(), "jahre": frei_je_jahr},
    },
}

def json_kompakt(objekt):
    """Wie json.dumps(indent=1), aber Listen aus Zahlen auf einer Zeile,
    damit die Profile die Datei nicht auf Tausende Zeilen aufblähen."""
    zeilen = json.dumps(objekt, ensure_ascii=False, indent=1).split("\n")
    ausgabe, puffer = [], None
    for zeile in zeilen:
        inhalt = zeile.strip()
        if puffer is None and inhalt.endswith("[") and not inhalt.endswith("[]"):
            puffer = [zeile]
            continue
        if puffer is not None:
            if inhalt.rstrip(",").lstrip("-").replace(".", "", 1).isdigit():
                puffer.append(inhalt)
                continue
            if inhalt in ("]", "],") and len(puffer) > 1:
                ausgabe.append(puffer[0] + " ".join(puffer[1:]) + inhalt)
                puffer = None
                continue
            ausgabe.extend(puffer)
            puffer = None
        ausgabe.append(zeile)
    return "\n".join(ausgabe) + "\n"

with open(JSON_PFAD, "w", encoding="utf-8") as f:
    f.write(json_kompakt(ausgabe))

print("geschrieben:", JSON_PFAD)
print("Jahre gesamt:", [j["jahr"] for j in werktage_je_jahr])

geschrieben: data/stillste-tage.json
Jahre gesamt: [2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026]


## 11. Seite speisen

Die Jahresauswahl bekommt je Jahr einen Punkt auf der Höhe des
Werktagsmedians (relativ zum höchsten Jahr), der den Rückgang über die
Jahre zeigt.

`fetch()` auf eine lokale JSON-Datei wird von Browsern aus dem Dateisystem
blockiert. Damit `index.html` auch offline und ohne Server läuft, wird der
JSON-Inhalt zwischen Markierungen in die Seite eingebettet. Zusätzlich
wird die Liste aller Jahre als statisches HTML eingesetzt, damit die Seite
ohne JavaScript dieselben Tage mit sichtbarem Datum zeigt. Die Markierungen
bleiben stehen; der Rest der Seite wird nicht angefasst.

In [12]:
HTML_PFAD = "index.html"
WOCHENTAGE_LANG = ["Montag", "Dienstag", "Mittwoch", "Donnerstag", "Freitag", "Samstag", "Sonntag"]
MONATE_LANG = ["Januar", "Februar", "März", "April", "Mai", "Juni", "Juli",
               "August", "September", "Oktober", "November", "Dezember"]

def zahl(n):
    """Ganzzahl mit schmalem Leerzeichen als Tausendertrenner; rundet wie
    JavaScripts Math.round (x.5 aufwärts), damit Seite und Skript übereinstimmen."""
    return f"{int(n + 0.5):,}".replace(",", "\u202f")

def datum_lang(iso, mit_wochentag=True):
    d = datetime.date.fromisoformat(iso)
    text = f"{d.day}. {MONATE_LANG[d.month - 1]} {d.year}"
    return f"{WOCHENTAGE_LANG[d.weekday()]}, {text}" if mit_wochentag else text

WOCHENTAGE_KURZ = ["Mo", "Di", "Mi", "Do", "Fr", "Sa", "So"]

def svg_profil(profil, typisch):
    """Tagesverlauf als Linie, dahinter das typische Werktagsprofil.
    Ganzzahlige Koordinaten, damit Python und JavaScript dasselbe Markup erzeugen."""
    hoechst = max(max(profil), max(typisch))
    def punkte(werte):
        n = len(werte)
        return " ".join(f"{int(i * 950 / (n - 1) + 0.5)},{int(400 - v * 400 / hoechst + 0.5)}"
                        for i, v in enumerate(werte))
    return ('<svg class="profil" viewBox="0 0 950 400" preserveAspectRatio="none" aria-hidden="true">'
            '<rect class="null" x="0" y="399" width="950" height="1"/>'
            f'<polyline class="typisch" points="{punkte(typisch)}"/>'
            f'<polyline class="tag-linie" points="{punkte(profil)}"/></svg>')

def svg_woche(woche, datum):
    """Sieben Balken Montag bis Sonntag, der aufgedeckte Tag hervorgehoben."""
    teile = ['<svg class="woche" viewBox="0 0 70 30" preserveAspectRatio="none" aria-hidden="true">']
    for i, w in enumerate(woche):
        if w["prozent"] is None:
            continue
        h = int(min(w["prozent"], 120) / 4 + 0.5)
        klasse = ' class="ist"' if w["datum"] == datum else ""
        teile.append(f'<rect{klasse} x="{i * 10 + 1}" y="{30 - h}" width="8" height="{h}"/>')
    teile.append('</svg>')
    return "".join(teile)

def html_jahr(jd, segment="gesamt", balken="var(--balken-b)"):
    teile = [f'<section class="jahr" id="jahr-{jd["jahr"]}" style="--balken: {balken}">',
             f'<h2>{jd["jahr"]}</h2>']
    info = f'{jd["tage"]} Werktage, Werktagsmedian {zahl(jd["median"])}.'
    if not jd["vollstaendig"]:
        info += f' Unvollständiges Jahr, Daten bis {datum_lang(jd["letzter_tag"], False)}.'
    teile.append(f'<p class="info">{info}</p>')
    teile.append('<p class="info">Balken: Tageswert in Prozent des Werktagsmedians, '
                 'Striche bei 50 und 75\u202f%, Spurende ist 100\u202f%.</p>')
    teile.append('<ol class="tage">')
    for t in jd["stillste"]:
        tid = f't-{segment}-{jd["jahr"]}-{t["rang"]}'
        teile.append('<li class="tag">')
        teile.append(f'<button type="button" aria-expanded="false" aria-controls="{tid}">')
        teile.append(f'<span class="rang">{t["rang"]}</span>')
        teile.append('<svg viewBox="0 0 100 8" preserveAspectRatio="none" aria-hidden="true">'
                     '<rect class="spur" x="0" y="0" width="100" height="8"/>'
                     f'<rect class="wert" x="0" y="0" width="{t["prozent"]}" height="8"/>'
                     '<rect class="strich" x="50" y="0" width="0.4" height="8"/>'
                     '<rect class="strich" x="75" y="0" width="0.4" height="8"/></svg>')
        teile.append(f'<span class="prozent">{zahl(t["prozent"])}\u202f%</span>')
        teile.append('<span class="hinweis">des Werktagsmedians, Datum aufdecken</span>')
        teile.append('</button>')
        klasse = "datum mit-feiertag" if t.get("feiertag") else "datum"
        teile.append(f'<div class="{klasse}" id="{tid}">')
        if t.get("feiertag"):
            teile.append(f'<span class="feiertag">{t["feiertag"]}</span>')
        teile.append(f'<time datetime="{t["datum"]}">{datum_lang(t["datum"])}</time>')
        teile.append(f'<span class="absolut">Tageswert {zahl(t["wert"])}, {t["intervalle"]} Intervalle</span>')
        if t["interpoliert"]:
            teile.append('<span class="interpoliert">Enthält einen laut Herausgeber interpolierten Zeitstempel.</span>')
        teile.append('<div class="bilder">')
        teile.append('<figure><figcaption>Tagesverlauf, blass: typischer Werktag</figcaption>'
                     + svg_profil(t["profil"], jd["profil_typisch"]) + '</figure>')
        teile.append('<figure><figcaption>Die Woche, Montag bis Sonntag</figcaption>'
                     + svg_woche(t["woche"], t["datum"])
                     + '<span class="wtage">' + "".join(f"<i>{w}</i>" for w in WOCHENTAGE_KURZ) + '</span></figure>')
        teile.append('</div>')
        teile.append('</div></li>')
    teile.append('</ol></section>')
    return "\n".join(teile)

def ersetzen(html, name, inhalt, block=True):
    anfang, ende = f"<!-- {name}-ANFANG -->", f"<!-- {name}-ENDE -->"
    a, e = html.index(anfang) + len(anfang), html.index(ende)
    trenner = "\n" if block else ""
    return html[:a] + trenner + inhalt + trenner + html[e:]

with open(JSON_PFAD, encoding="utf-8") as f:
    daten = json.load(f)

with open(HTML_PFAD, encoding="utf-8") as f:
    html = f.read()

hoechster_median = max(j["median"] for j in daten["gesamt"])
html = ersetzen(html, "JAHRE", "\n".join(
    f'    <li><a href="#jahr-{j["jahr"]}" title="Werktagsmedian {zahl(j["median"])}">'
    f'<span>{j["jahr"]}</span><span class="trend" style="--h: {j["median"] / hoechster_median:.3f}"></span></a></li>'
    for j in daten["gesamt"]))
html = ersetzen(html, "LISTE", "\n\n".join(html_jahr(j) for j in daten["gesamt"]))
html = ersetzen(html, "STAND", datum_lang(daten["stand"], False), block=False)
html = ersetzen(html, "DATEN", json.dumps(daten, ensure_ascii=False, separators=(",", ":")).replace("</", "<\\/"))

with open(HTML_PFAD, "w", encoding="utf-8") as f:
    f.write(html)

print("index.html aktualisiert:", len(html), "Zeichen,", len(daten["gesamt"]), "Jahre eingebettet")

index.html aktualisiert: 800701 Zeichen, 15 Jahre eingebettet
